# 🎨 Agentic Design Patterns with LangGraph (Python)

## 📋 Learning Objectives

This notebook demonstrates essential design patterns for building intelligent agents using LangGraph with Azure OpenAI integration. You'll learn proven patterns and architectural approaches that make agents more robust, maintainable, and effective using LangGraph's state management and workflow orchestration.

**Core Design Patterns Covered:**
- 🏗️ **Graph Factory Pattern**: Standardized graph creation and configuration
- 🔧 **Node Registry Pattern**: Organized approach to managing agent capabilities
- 🧵 **State Management**: Effective patterns for multi-turn interactions
- 🔄 **Workflow Orchestration**: Best practices for handling agent execution flows

## 🎯 Key Architectural Concepts

### Design Principles
- **Separation of Concerns**: Clear boundaries between graph logic, nodes, and state
- **Composability**: Building complex workflows from reusable components
- **Extensibility**: Patterns that allow easy addition of new capabilities
- **Testability**: Design for easy unit testing and validation

### LangGraph Integration
- **State Management**: Centralized state handling across workflow steps
- **Node Orchestration**: Defining and connecting workflow nodes
- **Conditional Routing**: Dynamic flow control based on state and conditions
- **Error Recovery**: Robust error handling and retry patterns
- **Human-in-the-Loop**: Approval and intervention patterns

## 🔧 Technical Architecture

### Core Components
- **LangGraph**: Python implementation for building stateful, multi-actor applications
- **Azure OpenAI API**: Access to state-of-the-art language models via secure endpoints
- **State Graph Pattern**: Centralized state management across workflow execution
- **Environment Configuration**: Secure and flexible configuration management

### Design Pattern Benefits
- **Maintainability**: Clear workflow organization and structure
- **Scalability**: Patterns that grow with your application needs
- **Reliability**: Proven approaches that handle edge cases
- **Performance**: Efficient resource utilization and API usage

## ⚙️ Prerequisites & Setup

**Required Dependencies:**
```bash
pip install langgraph langchain-openai langchain-core python-dotenv
```

**Environment Configuration (.env file):**
```env
AZURE_OPENAI_API_KEY=your_azure_openai_key
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com
AZURE_OPENAI_MODEL=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-08-01-preview
```

**Azure OpenAI Access:**
- Azure subscription with Azure OpenAI access approved
- Deployed model (deployment name referenced by `AZURE_OPENAI_MODEL`)
- Properly scoped API key OR Azure Active Directory auth
- Awareness of quota and rate limits for your pricing tier

## 📚 Design Pattern Categories

### 1. **Creational Patterns**
- Graph factory and builder patterns
- Configuration management patterns
- Dependency injection for graph services

### 2. **Behavioral Patterns**
- Node execution and orchestration
- State flow management
- Conditional routing and decision making

### 3. **Integration Patterns**
- Azure OpenAI endpoint integration
- Error handling and retry logic
- Resource management and cleanup

## 🚀 Best Practices Demonstrated

- **Clean Architecture**: Layered design with clear responsibilities
- **Error Handling**: Comprehensive exception management
- **Configuration**: Environment-based setup for different environments
- **Testing**: Patterns that enable effective unit and integration testing
- **Documentation**: Self-documenting code with clear intent

Ready to explore professional agent design patterns with LangGraph? Let's build something robust! 🌟

In [2]:
# 📦 Import Core Libraries for LangGraph Agent Design Patterns
import os                     # Environment variable access for configuration management
from random import randint    # Random selection utilities for tool functionality
from typing import Dict, List, TypedDict, Annotated
import operator               # For state reduction operations

from dotenv import load_dotenv  # Secure environment configuration loading

In [3]:
# 🤖 Import LangGraph and LangChain Components  
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig

In [4]:
# 🔧 Configuration Loading Pattern
# Loads environment variables for Azure OpenAI model deployments and other config.
# Follows external configuration principle for cloud-native applications.
load_dotenv()

True

In [5]:
# 🛠️ Tool Function Design Pattern
# Implements the Strategy Pattern for pluggable agent capabilities.
# Demonstrates separation of business logic from agent orchestration.

@tool
def get_random_destination() -> str:
    """Get a random vacation destination.
    
    Patterns illustrated:
    - Strategy Pattern: Interchangeable selection algorithm
    - Repository Pattern: Encapsulated data source
    - Factory Method: Creates destination objects on demand
    
    Returns:
        str: A randomly selected destination.
    """
    destinations = [
        "Barcelona, Spain",
        "Paris, France",
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    ]
    return destinations[randint(0, len(destinations) - 1)]

In [6]:
# 🏗️ State Management Pattern
# Define the agent state structure using TypedDict
class TravelAgentState(TypedDict):
    """State structure for our Travel Agent graph."""
    messages: Annotated[List[HumanMessage | AIMessage | SystemMessage], operator.add]
    user_preferences: Dict[str, any]
    conversation_history: List[Dict[str, any]]
    current_destination: str
    confidence_score: float

In [7]:
azure_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
azure_api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
# Support either AZURE_AI_FOUNDRY_MODEL_ID or legacy AZURE_AI_FOUNDRY_MODEL
azure_deployment = os.getenv("AZURE_AI_FOUNDRY_MODEL_ID") or os.getenv("AZURE_AI_FOUNDRY_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-07-01-preview")

# Debug prints (mask key)
print(f"Azure Endpoint: {azure_endpoint}")
print(f"Azure Deployment: {azure_deployment}")
print(f"API Version: {api_version}")

Azure Endpoint: https://ibecfoundry.openai.azure.com/
Azure Deployment: gpt-4o
API Version: 2024-02-01


In [8]:
# Initialize Azure OpenAI chat model
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=api_version,
    deployment_name=azure_deployment,
    temperature=0.7
)


In [9]:
# ✅ Quick sanity check: call the raw model before building the agent
from langchain_core.messages import HumanMessage

try:
    test_resp = llm.invoke([HumanMessage(content="Return just the word TEST")])
    print("Model test OK ->", test_resp.content)
except Exception as e:
    print("Model test failed:", e)
    raise

Model test OK -> TEST


In [10]:
# 🎯 Agent Instructions and Configuration
AGENT_NAME = "TravelAgent"

SYSTEM_MESSAGE = """You are a helpful AI Travel Agent that can help plan vacations for customers.

Important: When users specify a destination, always plan for that location. Only suggest random destinations when the user hasn't specified a preference.

When the conversation begins, introduce yourself with this message:
"Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?"

Always prioritize user preferences. If they mention a specific destination like "Bali" or "Paris," focus your planning on that location rather than suggesting alternatives.
"""

In [11]:
# 🔧 Node Functions for LangGraph Workflow

def call_model(state: TravelAgentState) -> TravelAgentState:
    """Node that calls the LLM with current state."""
    messages = state["messages"]
    
    # Add system message if this is the start of conversation
    if len(messages) == 1 and isinstance(messages[0], HumanMessage):
        system_msg = SystemMessage(content=SYSTEM_MESSAGE)
        messages = [system_msg] + messages
    
    # Bind tools to the model
    model_with_tools = llm.bind_tools([get_random_destination])
    response = model_with_tools.invoke(messages)
    
    return {"messages": [response]}


def should_continue(state: TravelAgentState) -> str:
    """Conditional edge function to determine next step."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM makes a tool call, route to tools
    if last_message.tool_calls:
        return "tools"
    # Otherwise, end the workflow
    return END

In [12]:
# 🏗️ Graph Factory Pattern - Create the Travel Agent Workflow
def create_travel_agent_graph() -> StateGraph:
    """Factory function to create a configured travel agent graph."""
    
    # Initialize the graph with our state structure
    workflow = StateGraph(TravelAgentState)
    
    # Add nodes to the graph
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", ToolNode([get_random_destination]))
    
    # Set the entry point
    workflow.set_entry_point("agent")
    
    # Add conditional edges
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "tools": "tools",
            END: END
        }
    )
    
    # After tools, always go back to agent
    workflow.add_edge("tools", "agent")
    
    return workflow

# Create the graph
graph = create_travel_agent_graph()

# Compile with memory saver for conversation persistence
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("✅ Travel Agent Graph created successfully!")

✅ Travel Agent Graph created successfully!


In [13]:
# 🧪 Test the Basic Travel Agent

# Create a conversation thread
thread_config = {"configurable": {"thread_id": "travel_conversation_1"}}

# Initial state
initial_state = {
    "messages": [HumanMessage(content="Plan me a day trip")],
    "user_preferences": {},
    "conversation_history": [],
    "current_destination": "",
    "confidence_score": 0.0
}

print("🚀 Testing basic travel agent...")
result = app.invoke(initial_state, config=thread_config)

# Extract and display the response
last_message = result["messages"][-1]
print("\n🧳 Travel Agent Response:")
print(last_message.content)

🚀 Testing basic travel agent...

🧳 Travel Agent Response:
"Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?"

Do you already have a specific destination in mind for your day trip? If not, I can suggest one for you!


In [14]:
# 🔄 Continue the conversation
follow_up_state = {
    "messages": [HumanMessage(content="I don't like that destination. Plan me another vacation.")],
    "user_preferences": {},
    "conversation_history": [],
    "current_destination": "",
    "confidence_score": 0.0
}

print("🔄 Testing conversation continuation...")
result2 = app.invoke(follow_up_state, config=thread_config)

# Extract and display the response
last_message2 = result2["messages"][-1]
print("\n🔄 Follow-up Response:")
print(last_message2.content)

🔄 Testing conversation continuation...

🔄 Follow-up Response:
It seems the random destination generator picked the same location twice: Paris, France. Would you like me to try again for a different destination, or should I plan your day trip to Paris?


# 🧩 Workshop: Implementing Agentic Design Principles with LangGraph

Now that we have a basic Travel Agent working with LangGraph, let's enhance it with the **Agentic Design Principles** from the lesson README. We'll work through practical exercises that build upon our existing graph structure.

## 📝 Workshop Structure

Each task will:
1. **Explain** the design principle and why it matters
2. **Implement** the pattern using our LangGraph Travel Agent as the foundation
3. **Test** the enhancement with practical examples
4. **Reflect** on how this improves the user experience

Let's transform our simple graph into one that embodies human-centric design principles!

## 🛰️ Task 1: Agent (Space) - Connecting People, Events & Knowledge

**Principle:** "Connecting, not collapsing" + "Easily accessible yet occasionally invisible"

**Challenge:** Our Travel Agent currently works in isolation. Let's make it connect users to relevant people, events, and knowledge while remaining unobtrusive.

**What we'll build:**
- A connection system that links destinations to events, people, and local knowledge
- Smart nudging that only activates when truly helpful
- Background discovery that enhances responses without being pushy

**Implementation:** We'll add new nodes to our graph that handle connection discovery and intelligent nudging.

In [16]:
# 🔧 Implementing Space Principle: Enhanced Travel Agent with Connections

# First, let's create a knowledge graph for travel connections
class TravelConnectionGraph:
    def __init__(self):
        self.connections = {
            # Events happening in destinations
            "Paris, France": {
                "events": ["Paris Fashion Week (Sept)", "Louvre Night Tours", "Seine River Festival"],
                "local_experts": ["Marie (food blogger)", "Jacques (history guide)"],
                "hidden_gems": ["Covered Passages", "Promenade Plantée", "Père Lachaise Cemetery"]
            },
            "Tokyo, Japan": {
                "events": ["Cherry Blossom Festival", "Tokyo Game Show", "Sumida River Fireworks"],
                "local_experts": ["Yuki (temple guide)", "Hiroshi (food critic)"],
                "hidden_gems": ["Golden Gai", "Yanaka District", "Robot Restaurant"]
            },
            "Barcelona, Spain": {
                "events": ["La Mercè Festival", "Barcelona Beach Festival", "Gaudí Architecture Tours"],
                "local_experts": ["Carlos (architecture expert)", "Isabella (flamenco dancer)"],
                "hidden_gems": ["Park Güell at sunrise", "El Born Cultural Center", "Bunkers del Carmel"]
            }
        }
    
    def get_connections(self, destination: str) -> dict:
        return self.connections.get(destination, {})
    
    def should_suggest_connections(self, destination: str, user_interests: list = None) -> bool:
        """Only suggest connections if they're truly relevant"""
        connections = self.get_connections(destination)
        if not connections:
            return False
        
        # Smart logic: suggest if we have rich connections for this destination
        total_items = len(connections.get("events", [])) + len(connections.get("hidden_gems", []))
        return total_items >= 3

# Create our enhanced travel graph
travel_graph = TravelConnectionGraph()

# Enhanced tool that leverages connections
@tool
def get_enhanced_destination() -> dict:
    """Get a destination with rich connection data"""
    base_destination = get_random_destination.invoke({})
    connections = travel_graph.get_connections(base_destination)
    
    return {
        "destination": base_destination,
        "has_connections": bool(connections),
        "connections": connections,
        "connection_count": len(connections.get("events", [])) + len(connections.get("hidden_gems", []))
    }

# Test the enhancement
enhanced_dest = get_enhanced_destination.invoke({})
print(f"Enhanced destination selection: {enhanced_dest}")

# Show connections for a specific destination
paris_connections = travel_graph.get_connections("Paris, France")
print(f"\nParis connections: {paris_connections}")

Enhanced destination selection: {'destination': 'Sydney, Australia', 'has_connections': False, 'connections': {}, 'connection_count': 0}

Paris connections: {'events': ['Paris Fashion Week (Sept)', 'Louvre Night Tours', 'Seine River Festival'], 'local_experts': ['Marie (food blogger)', 'Jacques (history guide)'], 'hidden_gems': ['Covered Passages', 'Promenade Plantée', 'Père Lachaise Cemetery']}


## ⏳ Task 2: Agent (Time) - Learning from Past, Present, Future

**Principle:** Reflect on history, nudge intelligently in the present, adapt for the future

**Challenge:** Our agent treats each conversation as brand new. Let's make it learn from past interactions, provide contextual nudges, and adapt its behavior over time.

**What we'll build:**
- Memory system that remembers user preferences and past trips
- Smart nudging based on context rather than generic notifications
- Adaptation mechanism that personalizes responses based on feedback

**Implementation:** We'll enhance our state structure and add memory-aware nodes to the graph.

In [17]:
# 🔧 Implementing Time Principle: Memory-Enhanced Travel Agent

from datetime import datetime

# Enhanced State Structure with Memory
class EnhancedTravelAgentState(TypedDict):
    """Enhanced state structure with memory capabilities."""
    messages: Annotated[List[HumanMessage | AIMessage | SystemMessage], operator.add]
    user_preferences: Dict[str, any]
    conversation_history: List[Dict[str, any]]
    current_destination: str
    confidence_score: float
    memory_context: str
    nudges: List[str]

# Memory system for our Travel Agent
class TravelMemory:
    def __init__(self):
        self.user_preferences = {}
        self.conversation_history = []
        self.successful_suggestions = []
        
    def record_interaction(self, user_input: str, agent_response: str, user_feedback: str = None):
        """Record each interaction for learning"""
        interaction = {
            "timestamp": datetime.now().isoformat(),
            "user_input": user_input,
            "agent_response": agent_response,
            "user_feedback": user_feedback
        }
        self.conversation_history.append(interaction)
        
        # Extract preferences from user input
        self._extract_preferences(user_input, user_feedback)
    
    def _extract_preferences(self, user_input: str, feedback: str = None):
        """Learn user preferences from their inputs and feedback"""
        input_lower = user_input.lower()
        
        # Extract destination type preferences
        if "beach" in input_lower:
            self.user_preferences["prefers_beaches"] = True
        if "mountain" in input_lower:
            self.user_preferences["prefers_mountains"] = True
        if "city" in input_lower or "urban" in input_lower:
            self.user_preferences["prefers_cities"] = True
        if "historical" in input_lower or "history" in input_lower:
            self.user_preferences["likes_history"] = True
        
        # Learn from feedback
        if feedback:
            feedback_lower = feedback.lower()
            if "too long" in feedback_lower or "shorter" in feedback_lower:
                self.user_preferences["prefers_concise"] = True
            if "more detail" in feedback_lower:
                self.user_preferences["wants_detail"] = True
    
    def get_relevant_context(self, current_request: str) -> str:
        """Get relevant past context for current request"""
        if not self.conversation_history:
            return ""
        
        # Simple relevance: look for similar keywords in past conversations
        request_words = set(current_request.lower().split())
        relevant_interactions = []
        
        for interaction in self.conversation_history[-5:]:  # Last 5 interactions
            past_words = set(interaction["user_input"].lower().split())
            if request_words.intersection(past_words):
                relevant_interactions.append(interaction)
        
        if relevant_interactions:
            return f"Based on our previous conversations, I remember you were interested in {', '.join(self.user_preferences.keys())}"
        return ""

# Create memory system for our agent
travel_memory = TravelMemory()

# Memory-aware node for the graph
def process_memory(state: EnhancedTravelAgentState) -> EnhancedTravelAgentState:
    """Node that processes memory and context for personalization"""
    messages = state["messages"]
    if messages:
        last_user_message = None
        for msg in reversed(messages):
            if isinstance(msg, HumanMessage):
                last_user_message = msg.content
                break
        
        if last_user_message:
            context = travel_memory.get_relevant_context(last_user_message)
            return {"memory_context": context, "user_preferences": travel_memory.user_preferences}
    
    return {"memory_context": "", "user_preferences": {}}

# Test memory system
travel_memory.record_interaction(
    "I want a beach vacation", 
    "I suggest Bali for beautiful beaches",
    "That sounds perfect!"
)

travel_memory.record_interaction(
    "Can you suggest something shorter?",
    "Brief suggestion: Try Nice, France",
    "Much better"
)

print("Learned preferences:", travel_memory.user_preferences)
print("Memory context for 'beach vacation':", travel_memory.get_relevant_context("I want a beach vacation"))

Learned preferences: {'prefers_beaches': True}
Memory context for 'beach vacation': Based on our previous conversations, I remember you were interested in prefers_beaches


## 🔐 Task 3: Agent (Core) - Building Trust Through Transparency

**Principle:** "Embrace uncertainty but establish trust" + Transparency + Control

**Challenge:** Users need to trust our agent's recommendations. Let's make it transparent about its confidence levels, show its reasoning process, and give users control over its behavior.

**What we'll build:**
- Confidence scoring system that shows uncertainty
- Activity log that shows what tools were used and why
- User control panel for customizing agent behavior
- Transparent reasoning that users can inspect

**Implementation:** We'll add transparency and control nodes to our LangGraph workflow.

In [18]:
# 🔧 Implementing Core Principle: Transparent & Controllable Travel Agent

# Agent Activity Logger - shows what the agent is doing
class AgentActivityLogger:
    def __init__(self):
        self.activities = []
    
    def log_tool_use(self, tool_name: str, purpose: str, result_summary: str):
        """Log when and why tools are used"""
        self.activities.append({
            "timestamp": datetime.now().isoformat(),
            "type": "tool_use",
            "tool": tool_name,
            "purpose": purpose,
            "result": result_summary[:100]  # Truncate for display
        })
    
    def log_node_execution(self, node_name: str, purpose: str, state_changes: str):
        """Log node execution in the graph"""
        self.activities.append({
            "timestamp": datetime.now().isoformat(),
            "type": "node_execution",
            "node": node_name,
            "purpose": purpose,
            "changes": state_changes[:100]
        })
    
    def get_recent_activities(self, count: int = 5) -> list:
        """Get recent activities for transparency"""
        return self.activities[-count:]

# Confidence scoring for travel recommendations
class ConfidenceCalculator:
    @staticmethod
    def calculate_destination_confidence(destination: str, user_context: dict) -> float:
        """Calculate confidence in a destination recommendation"""
        confidence = 0.5  # Base confidence
        
        # Higher confidence if we have rich data about destination
        if destination in travel_graph.connections:
            confidence += 0.3
        
        # Higher confidence if it matches user preferences
        if user_context.get("prefers_beaches") and "beach" in destination.lower():
            confidence += 0.2
        if user_context.get("likes_history") and destination in ["Paris, France", "Cairo, Egypt"]:
            confidence += 0.2
        
        return min(confidence, 1.0)  # Cap at 1.0
    
    @staticmethod
    def confidence_to_language(score: float) -> str:
        """Convert confidence score to human language"""
        if score >= 0.8:
            return "I'm very confident this is a great match"
        elif score >= 0.6:
            return "I think this would be a good choice"
        elif score >= 0.4:
            return "This might work for you"
        else:
            return "I'm less certain about this suggestion"

# User Control Settings
class TravelAgentController:
    def __init__(self):
        self.settings = {
            "verbosity": "normal",  # concise, normal, detailed
            "show_confidence": True,
            "show_reasoning": False,
            "auto_suggest_connections": True,
            "remember_preferences": True,
            "max_suggestions": 3
        }
    
    def update_settings(self, **updates):
        """Allow user to control agent behavior"""
        for key, value in updates.items():
            if key in self.settings:
                self.settings[key] = value
                print(f"✓ Updated {key} to {value}")
            else:
                print(f"✗ Unknown setting: {key}")
    
    def get_user_control_panel(self) -> str:
        """Show available controls to user"""
        controls = []
        for setting, value in self.settings.items():
            controls.append(f"  {setting}: {value}")
        return "🎛️ Agent Controls:\n" + "\n".join(controls)

# Initialize our enhanced systems
activity_logger = AgentActivityLogger()
confidence_calc = ConfidenceCalculator()
agent_controller = TravelAgentController()

# Transparency node for the graph
def calculate_transparency(state: EnhancedTravelAgentState) -> EnhancedTravelAgentState:
    """Node that calculates confidence and logs activities"""
    current_dest = state.get("current_destination", "")
    user_prefs = state.get("user_preferences", {})
    
    if current_dest:
        confidence = confidence_calc.calculate_destination_confidence(current_dest, user_prefs)
        activity_logger.log_node_execution(
            "transparency",
            "Calculating confidence score",
            f"Confidence: {confidence:.2f} for {current_dest}"
        )
        return {"confidence_score": confidence}
    
    return {"confidence_score": 0.5}

# Test the transparent system
print("🎛️ User Control Panel:")
print(agent_controller.get_user_control_panel())

print("\n🔍 Testing transparency calculation:")
test_state = {
    "messages": [],
    "user_preferences": {"prefers_beaches": True, "likes_history": True},
    "conversation_history": [],
    "current_destination": "Paris, France",
    "confidence_score": 0.0,
    "memory_context": "",
    "nudges": []
}

result = calculate_transparency(test_state)
print(f"Confidence Score: {result['confidence_score']:.2f}")
print(f"Confidence Explanation: {confidence_calc.confidence_to_language(result['confidence_score'])}")

print("\n📋 Recent Agent Activities:")
for activity in activity_logger.get_recent_activities():
    print(f"  {activity['type']}: {activity.get('node', activity.get('tool', 'unknown'))}")

🎛️ User Control Panel:
🎛️ Agent Controls:
  verbosity: normal
  show_confidence: True
  show_reasoning: False
  auto_suggest_connections: True
  remember_preferences: True
  max_suggestions: 3

🔍 Testing transparency calculation:
Confidence Score: 1.00
Confidence Explanation: I'm very confident this is a great match

📋 Recent Agent Activities:
  node_execution: transparency


## 🎛️ Task 4: Putting It All Together - Enhanced LangGraph Travel Agent

**Final Integration:** Now let's create an enhanced version of our Travel Agent graph that incorporates all the design principles we've implemented.

**What we'll create:**
- A new graph that uses our connection system, memory, and transparency features
- Demonstration of how all principles work together in a LangGraph workflow
- Before/after comparison showing the improvement

**Exercise:** Run both versions and compare how the enhanced graph provides a more human-centric experience.

In [19]:
# 🔧 Creating Enhanced Travel Agent Graph with All Design Principles

# Enhanced tool that incorporates all our principles
@tool
def get_enhanced_travel_suggestion(user_input: str) -> dict:
    """Enhanced travel suggestion incorporating all design principles"""
    
    # SPACE: Get destination with connection awareness
    base_dest = get_random_destination.invoke({})
    connections = travel_graph.get_connections(base_dest)
    
    # TIME: Use memory to personalize
    context = travel_memory.get_relevant_context(user_input)
    
    # CORE: Calculate confidence and log activity
    user_prefs = travel_memory.user_preferences
    confidence = confidence_calc.calculate_destination_confidence(base_dest, user_prefs)
    
    activity_logger.log_tool_use(
        "enhanced_travel_suggestion",
        f"Personalized suggestion based on: {user_input}",
        f"Suggested {base_dest} with {len(connections)} connections"
    )
    
    # CONSISTENCY: Format response uniformly
    response = {
        "destination": base_dest,
        "confidence": {
            "score": confidence,
            "explanation": confidence_calc.confidence_to_language(confidence)
        },
        "connections": connections,
        "personalization": context,
        "transparency": {
            "reasoning": f"Selected based on your preferences and available rich information",
            "tool_used": "enhanced_travel_suggestion",
            "memory_accessed": bool(context)
        }
    }
    
    return response

# Enhanced node functions
def enhanced_call_model(state: EnhancedTravelAgentState) -> EnhancedTravelAgentState:
    """Enhanced node that calls the LLM with enhanced context."""
    messages = state["messages"]
    memory_context = state.get("memory_context", "")
    user_prefs = state.get("user_preferences", {})
    
    # Enhanced system message with personalization
    enhanced_system_message = SYSTEM_MESSAGE
    if memory_context:
        enhanced_system_message += f"\n\nPersonalization context: {memory_context}"
    if user_prefs:
        enhanced_system_message += f"\n\nUser preferences: {user_prefs}"
    
    # Add system message if this is the start of conversation
    if len(messages) == 1 and isinstance(messages[0], HumanMessage):
        system_msg = SystemMessage(content=enhanced_system_message)
        messages = [system_msg] + messages
    
    # Bind enhanced tools to the model
    model_with_tools = llm.bind_tools([get_enhanced_travel_suggestion])
    response = model_with_tools.invoke(messages)
    
    return {"messages": [response]}

def enhanced_should_continue(state: EnhancedTravelAgentState) -> str:
    """Enhanced conditional edge function."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM makes a tool call, route to tools
    if last_message.tool_calls:
        return "tools"
    # Check if we should process memory
    elif not state.get("memory_context"):
        return "memory"
    # Otherwise, calculate transparency
    elif state.get("confidence_score", 0) == 0:
        return "transparency"
    # End the workflow
    return END

# Enhanced Graph Factory
def create_enhanced_travel_agent_graph() -> StateGraph:
    """Factory function to create an enhanced travel agent graph."""
    
    # Initialize the graph with our enhanced state structure
    workflow = StateGraph(EnhancedTravelAgentState)
    
    # Add nodes to the graph
    workflow.add_node("agent", enhanced_call_model)
    workflow.add_node("tools", ToolNode([get_enhanced_travel_suggestion]))
    workflow.add_node("memory", process_memory)
    workflow.add_node("transparency", calculate_transparency)
    
    # Set the entry point
    workflow.set_entry_point("memory")
    
    # Add edges
    workflow.add_edge("memory", "agent")
    workflow.add_edge("transparency", END)
    
    # Add conditional edges from agent
    workflow.add_conditional_edges(
        "agent",
        enhanced_should_continue,
        {
            "tools": "tools",
            "memory": "memory",
            "transparency": "transparency",
            END: END
        }
    )
    
    # After tools, go to transparency
    workflow.add_edge("tools", "transparency")
    
    return workflow

# Create the enhanced graph
enhanced_graph = create_enhanced_travel_agent_graph()
enhanced_app = enhanced_graph.compile(checkpointer=memory)

print("✅ Enhanced Travel Agent Graph created successfully!")
print("\n🆚 Comparison Summary:")
print("Original Graph: Basic LLM → Tools → End")
print("Enhanced Graph: Memory → LLM → Tools → Transparency → End")

# Demo the enhanced functionality
print("\n🔍 Enhanced graph demo:")
enhanced_state = {
    "messages": [HumanMessage(content="I want a cultural trip")],
    "user_preferences": {},
    "conversation_history": [],
    "current_destination": "",
    "confidence_score": 0.0,
    "memory_context": "",
    "nudges": []
}

enhanced_config = {"configurable": {"thread_id": "enhanced_travel_1"}}
enhanced_result = enhanced_app.invoke(enhanced_state, config=enhanced_config)

print(f"Final state keys: {list(enhanced_result.keys())}")
print(f"Confidence score: {enhanced_result.get('confidence_score', 'N/A')}")
print(f"Memory context available: {bool(enhanced_result.get('memory_context'))}")

✅ Enhanced Travel Agent Graph created successfully!

🆚 Comparison Summary:
Original Graph: Basic LLM → Tools → End
Enhanced Graph: Memory → LLM → Tools → Transparency → End

🔍 Enhanced graph demo:
Final state keys: ['messages', 'user_preferences', 'conversation_history', 'current_destination', 'confidence_score', 'memory_context', 'nudges']
Confidence score: 0.5
Memory context available: True


## 🎯 Workshop Reflection & Next Steps

**What We Accomplished:**

✅ **Agent (Space)**: Connected destinations to events, people, and knowledge using LangGraph nodes
✅ **Agent (Time)**: Added memory, learning, and adaptation through enhanced state management
✅ **Agent (Core)**: Built transparency, confidence scoring, and user control with dedicated graph nodes
✅ **Integration**: Created an enhanced graph that orchestrates all principles in a cohesive workflow

**Key LangGraph Patterns Demonstrated:**

1. **State Management**: Using TypedDict to define clear state structure across the workflow
2. **Node Composition**: Breaking functionality into focused, reusable nodes
3. **Conditional Routing**: Dynamic flow control based on state and business logic
4. **Memory Integration**: Persistent state across conversation turns using checkpointers
5. **Tool Orchestration**: Seamless integration of tools within the graph workflow

**Key Takeaways:**

1. **Human-Centric Design**: LangGraph's stateful approach naturally supports human-centered workflows
2. **Progressive Enhancement**: Graph structure allows incremental addition of capabilities
3. **Trust Through Transparency**: Dedicated nodes can handle transparency and confidence calculations
4. **Learning Over Time**: State persistence enables agents to evolve with user interactions

**Try These Exercises:**

1. **Test Both Graphs**: Compare responses from the original vs enhanced graph
2. **Experiment with Routing**: Modify conditional edges to change workflow behavior
3. **Add New Nodes**: Create additional nodes for specific functionalities
4. **Extend State**: Add new fields to the state structure for additional capabilities

**Next Level Enhancements:**

- Add human-in-the-loop approval nodes for sensitive operations
- Implement parallel processing nodes for efficiency
- Add error handling and recovery nodes
- Create sub-graphs for complex multi-step operations
- Build a web interface that visualizes the graph execution

**LangGraph vs Agent Framework Comparison:**

| Aspect | LangGraph | Agent Framework |
|--------|-----------|----------------|
| **State Management** | Explicit state flow through nodes | Implicit state in conversation |
| **Workflow Control** | Graph-based conditional routing | Linear tool execution |
| **Extensibility** | Add nodes and edges | Add tools and modify instructions |
| **Debugging** | Visual graph execution | Message-based inspection |
| **Complexity** | Higher (graph design) | Lower (instruction-based) |
| **Flexibility** | Very high (custom flows) | Medium (tool orchestration) |

**Remember**: LangGraph excels at complex, stateful workflows where you need fine-grained control over execution flow, while simpler agent frameworks are great for straightforward tool-calling patterns. Choose based on your complexity needs!